In [1]:
"""
Stock NeurIPS2018 Part 1. Data

This series is a reproduction of paper "Deep reinforcement learning for automated stock trading: An ensemble strategy".

Introduce how to use FinRL to fetch and process data that we need for ML/RL trading.
"""

from __future__ import annotations

import itertools

import pandas as pd
import yfinance as yf

from finrl.config import INDICATORS
from finrl.config import TRADE_END_DATE
from finrl.config import TRADE_START_DATE
from finrl.config import TRAIN_END_DATE
from finrl.config import TRAIN_START_DATE
from finrl.meta.preprocessor.preprocessors import FeatureEngineer
from finrl.meta.preprocessor.dow import DOW_30_HISTORICAL_UNIVERSE
from finrl.meta.preprocessor.dow import add_dow_membership
from finrl.meta.preprocessor.preprocessors import data_split
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader

# %% Part 1. Fetch data - Single ticker

# Using yfinance directly
aapl_df_yf = yf.download(tickers="aapl", start="2020-01-01", end="2020-01-31")
print("=== yfinance download ===")
print(aapl_df_yf.head())

# Using FinRL's YahooDownloader
aapl_df_finrl = YahooDownloader(
    start_date="2020-01-01",
    end_date="2020-01-31",
    ticker_list=["aapl"],
).fetch_data()
print("\n=== FinRL YahooDownloader ===")
print(aapl_df_finrl.head())

# %% Part 2. Fetch data - DOW 30 tickers

print("\n=== Historical Dow ticker universe ===")
print(DOW_30_HISTORICAL_UNIVERSE)

df_raw = YahooDownloader(
    start_date=TRAIN_START_DATE,
    end_date=TRADE_END_DATE,
    ticker_list=list(DOW_30_HISTORICAL_UNIVERSE),
).fetch_data()
print("\n=== Raw data ===")
print(df_raw.head())

# %% Part 3. Preprocess data

fe = FeatureEngineer(
    use_technical_indicator=True,
    tech_indicator_list=INDICATORS,
    use_vix=True,
    use_turbulence=True,
    user_defined_feature=False,
    keep_incomplete_tickers=True,
)

processed = fe.preprocess_data(df_raw)

list_ticker = list(DOW_30_HISTORICAL_UNIVERSE)
list_date = list(
    pd.date_range(processed["date"].min(), processed["date"].max()).astype(str)
)
combination = list(itertools.product(list_date, list_ticker))

processed_full = pd.DataFrame(combination, columns=["date", "tic"]).merge(
    processed, on=["date", "tic"], how="left"
)
processed_full = processed_full[processed_full["date"].isin(processed["date"])]
processed_full = processed_full.sort_values(["date", "tic"])
processed_full = add_dow_membership(processed_full)
active_rows = processed_full["is_member"]
if processed_full.loc[active_rows, "close"].isna().any():
    missing = processed_full.loc[
        active_rows & processed_full["close"].isna(), ["date", "tic"]
    ]
    raise ValueError(
        "Yahoo Finance data is missing for active Dow constituents; "
        f"inspect these rows before training:\n{missing.head(10)}"
    )
feature_columns = [
    column
    for column in processed_full.columns
    if column not in {"date", "tic", "is_member"}
]
processed_full[feature_columns] = processed_full.groupby("tic", sort=False)[
    feature_columns
].transform(lambda values: values.ffill().bfill())
processed_full[feature_columns] = processed_full[feature_columns].fillna(0)
processed_full["is_member"] = processed_full["is_member"].astype(int)

print("\n=== Processed data ===")
print(processed_full.head())

# %% Part 4. Split and save data

train = data_split(processed_full, TRAIN_START_DATE, TRAIN_END_DATE)
trade = data_split(processed_full, TRADE_START_DATE, TRADE_END_DATE)
print(f"\nTrain data length: {len(train)}")
print(f"Trade data length: {len(trade)}")

train.to_csv("train_data.csv")
trade.to_csv("trade_data.csv")
print("Data saved to train_data.csv and trade_data.csv")


YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


=== yfinance download ===
Price           Close       High        Low       Open     Volume
Ticker           AAPL       AAPL       AAPL       AAPL       AAPL
Date                                                             
2020-01-02  72.271545  72.331702  71.029923  71.282575  135480400
2020-01-03  71.568932  72.326897  71.345153  71.501557  146322800
2020-01-06  72.139198  72.177699  70.442799  70.693051  118387200
2020-01-07  71.799889  72.403851  71.580920  72.148789  108872000
2020-01-08  72.954918  73.255698  71.503953  71.503953  132079200
YF deprecation warning: set proxy via new config function: yf.set_config(proxy=proxy)
Shape of DataFrame:  (20, 8)

=== FinRL YahooDownloader ===
Price        date      close       high        low       open     volume  \
0      2020-01-02  72.271545  72.331702  71.029923  71.282575  135480400   
1      2020-01-03  71.568932  72.326897  71.345153  71.501557  146322800   
2      2020-01-06  72.139198  72.177699  70.442799  70.693051  118387200

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['DWDP']: YFTzMissingError('possibly delisted; no timezone found')
[***************

Shape of DataFrame:  (115313, 8)

=== Raw data ===
Price        date       close        high         low        open     volume  \
0      2014-01-06   16.825546   16.914324   16.506005   16.625099  412610800   
1      2014-01-06   79.444374   80.452477   79.192347   80.137446    2838100   
2      2014-01-06   19.681499   19.850000   19.421000   19.792500   63412000   
3      2014-01-06   74.960457   75.420083   74.693040   74.960457    2844700   
4      2014-01-06  118.295090  119.448888  117.773740  119.141206    4196500   

Price   tic  day  
0      AAPL    0  
1      AMGN    0  
2      AMZN    0  
3       AXP    0  
4        BA    0  
Successfully added technical indicators


[*********************100%***********************]  1 of 1 completed


Shape of DataFrame:  (3068, 8)
Successfully added vix
Successfully added turbulence index


ValueError: Yahoo Finance data is missing for active Dow constituents; inspect these rows before training:
           date  tic
35   2014-01-06  UTX
76   2014-01-07  UTX
117  2014-01-08  UTX
158  2014-01-09  UTX
199  2014-01-10  UTX
322  2014-01-13  UTX
363  2014-01-14  UTX
404  2014-01-15  UTX
445  2014-01-16  UTX
486  2014-01-17  UTX